In [1]:
from copy import copy
from collections import defaultdict, Counter
import pathlib
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression, \
                                 LinearRegression
from sklearn.metrics import precision_recall_fscore_support

from pprint import pprint

# import conllu
from scipy.sparse import csr_matrix
import pymorphy3

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# from umap.umap_ import UMAP

In [2]:
morph = pymorphy3.MorphAnalyzer()
morpho_cashe = defaultdict(str)

# Расчёт статистики всех связей в фрагменте книги с учётом их направления.
def calc_syntax_stat(book):
    con_stat = defaultdict(int)
    overall = 0
    for sentence in book:
        overall += len(sentence)
        for child_position, word in enumerate(sentence):
            if word['head'] is None:
                continue
            parent_position = word['head'] - 1
            if parent_position == -1:
                continue
            if parent_position > child_position:
                d = '<-'
            else:
                d = '->'
            con_stat[f"({sentence[parent_position]['upos']}, {d}, {word['upos']}, {word['deprel']})"] += 1

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

def calc_homonymy_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            if morpho_cashe[word['form']] != '':
                con_stat[morpho_cashe[word['form']]] += 1
            else:
                res = morph.parse(word['form'])
                nfs = list(set(a.normal_form for a in res))
                poses = list(set(a.tag.POS for a in res))

                if len(nfs) > 1: # Ambigous by lemma
                    if len(poses) > 1:
                        con_stat['pos_lemma'] += 1
                        morpho_cashe[word['form']] = 'pos_lemma'
                    else:
                        con_stat['lemma'] += 1
                        morpho_cashe[word['form']] = 'lemma'
                else:
                    if len(poses) > 1:
                        con_stat['pos'] += 1
                        morpho_cashe[word['form']] = 'pos'
                    else:
                        if len(res) > 1:
                            con_stat['feat'] += 1
                            morpho_cashe[word['form']] = 'feat'
                        else:
                            con_stat['unambig'] += 1
                            morpho_cashe[word['form']] = 'unambig'


    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

# service_poses = {'ADVB', 'NPRO', 'PRED', 'PREP', 'CONJ', 'PRCL', 'INTJ'}
service_poses = {'ADV', 'PRON', 'ADP', 'CCONJ', 'SCONJ', 'PART', 'DET'}

def calc_service_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            if word['upos'] in service_poses:
                con_stat[f'{word["lemma"]}_{word["upos"]}'] +=1

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

def calc_char_stat(book):
    con_stat = defaultdict(int)
    overall = 0

    for sentence in book:
        overall += len(sentence)
        for word in sentence:
            res = Counter(word['form'])
            for c, f in res.items():
                con_stat[c] += f

    for key in con_stat.keys():
        con_stat[key] /= overall
    return con_stat

In [3]:
class CONLLUParser:
    '''
    Собственный класс для разбора CONLLU файлов.
    Потому что стандартный какой-то очень медленный.
    '''
    sentences: list # Список предложения текста, содержащих синтаксическую информацию.
    texts: list # Список предложений в текстовой форме.

    def __init__(self):
        self.sentences = []
        self.texts = []

    # Превращает кортеж в словарь слова.
    def __make_wordform(self, word):
        res = dict()
        res['id'] = int(word[0])
        res['form'] = word[1]
        res['lemma'] = word[2]
        res['upos'] = word[3]
        res['head'] = int(word[6])
        res['deprel'] = word[7]
        return res

    def __read_sentence(self, lines, pos):
        ''' Возвращает предложение из списка строк lines, записанных в формате UD.
            Предложение находится начиная со строки с номером pos.
        '''
        sent = []
        text = ""
        while pos < len(lines) and lines[pos] != '' and lines[pos] != '\n':
            try:
              if lines[pos][0] != '#':
                  sent.append(self.__make_wordform(lines[pos][:-2].split("\t")))
              elif lines[pos].startswith('# text = '):
                  text = lines[pos][9:]
            except:
              print(f"Exception: {lines[pos]}")
            pos += 1
        pos += 1
        return sent, text, pos

    # Считывает книгу в conllu из списка строк.
    def parse_conllu_lines(self, lines):
        del self.sentences
        del self.texts
        self.sentences = []
        self.texts = []
        pos = 0
        while pos < len(lines):
            sent, text, pos = self.__read_sentence(lines, pos)
            self.sentences.append(sent)
            self.texts.append(text)

    # Считывает книгу в conllu из файла.
    def parse_conllu_file(self, filename):
        with open(filename, "rt", encoding='utf-8') as book_file:
            lines = book_file.readlines()
            self.parse_conllu_lines(lines)
            del lines

In [4]:
class WriterStorage:
    '''
        Класс служит для расчёта показателей, по коорым будет определяться авторство.
      Из модуля calc_stat_vectors выгружаются функции, которые считают параметры:
      частоты по синтаксису, грамматической неоднозначности, символам, служебным частям речи.
      Он умеет сеарилизовать себя в json, потому что когда считаешь фрагментами по 10 предложений,
      по большой коллекции тебе мало 64Г оперативки.
    '''

    # Описание свойств
    book_authors: list # Список имён авторов по книгам.
    book_names: list # Список названий книг.
    authors_id: dict # Словарь соответствий фамилии автора и его идентификатора.
    book_author_id: list # Список идентификаторов авторов по книгам.
    parts_author_id: list # Список идентификаторов авторов по частям.
    book_no: list # Номер книги в списке.

    books_stat: list # Статистика по книгам в виде словаря.
    parts_stat: list # Статистика по частям в виде словаря.
    part_no: list # Номер части в книге.
    part_book_no: list # Номер книги соответствующей части.
    books_vect_dict: dict # Словарь признаков и их номеров в векторе по книгам в целом.
    books_vect: list # Векторы признаков по книгам в целом.
    parts_vect_dict: dict # Словарь признаков и их номеров в векторе по всем частям книг.
    parts_vect: list # Векторы признаков по частям книг.
    texts_by_parts: list # Тексты по частям.

    def __init__(self):
        self.book_names = []
        self.book_authors = []
        self.book_sentences = []
        self.authors_id = dict()

    # Словарь с функциями, коорые используются для расчёта. Ключ передается в функцию в качестве параметра.
    __func_correspondence  = {'syntax': calc_syntax_stat,
                              'service': calc_service_stat,
                              'homonymy': calc_homonymy_stat,
                              'char': calc_char_stat
                             }

    def calc_connections_distr(self, book, text, what='syntax', fragment_len = 100):
        '''
           Функция расчёта параметров для одной книги.
           book: Текст книги с синтаксической разметкой, разделенный на предложения.
           text: Текст книги, разделенный на предложения. Нужен, чтобы можно было посмореть где что.
           what: Какие параметры надо считать. Определяет какая функция расчёта параметров будет вызвана.
           fragment_len: Длина фрагментов, на кототрые надо делить.
           Возвращает синтаксически рамеченные предложения и текст книги, разделенные фрагменты.
           Если fragment_len = -1, возвращает книгу целиком.
        '''
        # Проверка, есть ли у нас такой ключ.
        if what not in WriterStorage.__func_correspondence.keys():
            raise Exception(f'We calculate statistics for the followig keys: '
                            f'{", ".join(WriterStorage.__func_correspondence.keys())}')

        # Расчёт статистики.
        # Определяем по какой функции ведётся расчёт.
        calc_connections_stat = WriterStorage.__func_correspondence[what]

        # -1 означает,что стаисика считаеттся для книги в целом.
        if fragment_len == -1:
            distr = calc_connections_stat(book)
            texts = [[text]]
        # В проивном случае надо разделить книгу на фрагменты.
        else:
            distr = []
            texts = []
            for i in range(0, len(book), fragment_len):
                # print(i, fragment_len)
                # print(book[i: i+fragment_len])
                distr.append(calc_connections_stat(book[i: i+fragment_len]))
                texts.append(text[i: i+fragment_len])
        return distr, texts

    def make_vector(self, data, keys):
        ''' Возвращает данные, собранные в виде словаря data в виде вектора,
            Соблюдая при этом порядок следования параметров, задаваемый при помощи keys.
        '''
        vect = np.zeros((len(keys)))
        for key, val in data.items():
            vect[keys[key]] = val
        return vect

    def vectorize_data(self, data):
        '''
        Векторизует данные из словаря data.
        Возвращает словарь признаков с номерами их позиций в векторе и сам вектор.
        '''
        data_dict = []
        for part in data:
            data_dict.extend(part.keys()) # data_dict |= set(part.keys())
        data_dict = list(set(data_dict))
        data_dict = {key:i for i, key in enumerate(data_dict)}

        # Теперь создаем разреженный массив с результирующими векторами.
        res_list = []
        rows = []
        cols = []
        for row_no, part in enumerate(data):
            res_list.extend(part.values())
            cols.extend([data_dict[key] for key in part.keys()])
            rows.extend([row_no for key in part.keys()])

        ret_data = csr_matrix((res_list, (rows, cols)), (len(data), len(data_dict))).toarray()

        # vects_a = []
        # for part in data:
        #     vects_a.append(self.make_vector(part, data_dict))
        return data_dict, ret_data

    def process_books(self, path, what='syntax', fragment_size=100):
        '''
        Функция обработки книг из каталога path по признакам, задаваемым параметром what.
        Режет текст на фрагменты размера fragment_size, если он равен -1, возвращает данные для книг целиком.
        '''
        parser = CONLLUParser() # Будет разбирать CONLLU.

        files2 = pathlib.Path(path).glob('*.conllu')
        self.books_stat = []
        self.parts_stat = []
        self.book_no = []
        self.part_no = []
        self.book_author_id = []
        self.parts_author_id = []
        self.authors_id = dict()
        self.part_book_no = []
        self.texts_by_parts = []
        self.fragment_size = fragment_size

        # Перебираем книги в CONLLU-формате.
        for i, file in enumerate(list(files2)):
            # Информация по книге. Выделяем из имени файла автора и название книги.
            # print(i, file.name)
            a = file.name.split("_")
            if a[0][-1] >= '0' and a[0][-1] <= '9':
                a[0] = a[0][:-1]
            self.book_authors.append(a[0])
            self.book_names.append(a[1][:-11])
            if a[0] not in self.authors_id.keys():
                auth_no = len(self.authors_id.keys())
                self.authors_id[a[0]] = auth_no
            else:
                auth_no = self.authors_id[a[0]]
            self.book_author_id.append(auth_no)
            self.book_no.append(i)
            print(i, a[0], a[1][:-11])

            # Читаем CONLLU.
            parser.parse_conllu_file(file.absolute())
            # book, text = parser.sentences, parser.texts
            # self.book_sentences.append(book)
            if fragment_size==-1:
                al, all_texts = self.calc_connections_distr(parser.sentences,  parser.texts, what, -1)
                self.books_stat.append(al)
            else:
                parts, parts_texts = self.calc_connections_distr(parser.sentences, parser.texts, what, fragment_size)
                # добавляем информацию и статисику по фрагментам.
                self.parts_stat.extend(parts)
                self.parts_author_id.extend([auth_no for part in parts])
                self.part_no.extend([p_n for p_n in range(len(parts))])
                self.part_book_no.extend([i for p_n in range(len(parts))])
                self.texts_by_parts.extend([t for t in parts_texts])

        # Векторизуем книги и фрагменты.
        if fragment_size == -1:
            d, v = self.vectorize_data(self.books_stat)
            self.books_vect_dict = d
            self.books_vect = v
        else:
            d, v = self.vectorize_data(self.parts_stat)
            self.parts_vect_dict = d
            self.parts_vect = v
        self.back_id = {i: a for a, i in self.authors_id.items()}

In [5]:
writers = WriterStorage()

writers.process_books('conllu_DeepPavlov', 'syntax', 500)

0 Abgaryan dalshe zhit
1 Abgaryan molchaie
2 Abgaryan 3apples
3 Abgaryan Fantastychysi
4 Abgaryan Jubileum
5 Abgaryan Ponaekhavshaya
6 Abgaryan Simon
7 Adamov Izgnanine vladyki
8 Adamov Pobediteli nedr
9 Adamov Tayna dvuh okeanov
10 Aksakov Vol1
11 Aksakov Vol2
12 Aksakov Vol3
13 Aksakov Vol4
14 Aksakov Vol5
15 AkuninBorisova Kreativschik
16 AkuninBrusnikin Bellona
17 AkuninBrusnikin Devyatny spas
18 AkuninBrusnikin Hero of another time
19 Akunin Almaznaya kolesnitsa 1
20 Akunin Almaznaya kolesnitsa 2
21 Akunin Azazel
22 Akunin Grom Pobedy
23 Akunin Leviafan
24 Akunin Lubovnik smerti
25 Akunin Nefritovye chetki
26 Akunin No West and East
27 Akunin Smert Ahillesa
28 Akunin Turetski gambit
29 Akunin World is theatre
30 Aldanov Armageddon
31 Aldanov Begstvo
32 Aldanov Bred
33 Aldanov Erfurtskoe svidanie
34 Aldanov Istoricheskie portrety
35 Aldanov Kartiny Oktyabrskogo perevorota
36 Aldanov Kluch
37 Aldanov Mogila voina
38 Aldanov Nachalo konca
39 Aldanov Ogon i dym
40 Aldanov Peschera
41 

#Переменные

In [6]:

#     book_authors: list # Список имён авторов по книгам.
#     book_names: list # Список названий книг.
#     authors_id: dict # Словарь соответствий фамилии автора и его идентификатора.
#     book_author_id: list # Список идентификаторов авторов по книгам.
#     parts_author_id: list # Список идентификаторов авторов по частям.

#     books_stat: list # Статистика по книгам в виде словаря.
#     parts_stat: list # Статистика по частям в виде словаря.
#     book_no: list # Номер книги в списке.
#     part_no: list # Номер части в книге.
#     books_vect_dict: dict # Словарь признаков и их номеров в векторе по книгам в целом.
#     books_vect: list # Векоры признаков по книгам в целом.
#     parts_vect_dict: dict # Словарь признаков и их номеров в векторе по всем частям книг.
#     parts_vect: list # Векоры признаков по частям книг.


parts_vect_np = np.array(writers.parts_vect)
parts_author_id_np = np.array(writers.parts_author_id)

# books_vect_np = np.array(writers.books_vect)
# books_author_id_np = np.array(writers.book_author_id)


In [7]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.feature_selection import f_classif
from collections import defaultdict

In [8]:
feature_names = [k for k, v in writers.parts_vect_dict.items()]
features = [v for k,v in writers.parts_vect_dict.items()]

In [9]:
parts_features = np.array(writers.parts_vect)
parts_author_ids = np.array(writers.parts_author_id)
author_ids = np.array(list(writers.authors_id.values()))
author_names = np.array(list(writers.authors_id.keys()))
author_id_to_name = {v: k for k, v in writers.authors_id.items()}

In [10]:
author_feature_means = []

for author_id in author_ids:
    author_mask = (parts_author_ids == author_id)
    author_feature_means.append(np.mean(parts_features[author_mask], axis=0))

feature_authors = np.array(author_feature_means)

In [11]:
def normalize_array(data):
    for i, array in enumerate(data):
        sum = np.sum(array)
        data[i] = array / sum
    return data

In [12]:
feature_authors = normalize_array(feature_authors)

##Фильтр фичей по пунктуации и частоте

In [13]:
punct_features = []

for author in feature_authors:
    for i, feature in enumerate(author):
        if "PUNCT" in feature_names[i]:
            punct_features.append(i)

In [14]:
notpunct_features = [i not in punct_features for i in range(feature_authors.shape[1])]

In [15]:
counter_features = Counter()

In [16]:
for author in feature_authors:
    for i, feature in enumerate(author):
        if feature > 1e-05:
          counter_features[i]+=1
        else:
          counter_features[i] += 0

# common_mask = [count != 0 and count < len(feature_authors)-50 for count in counter_features.values()]
common_mask = [count != 0 and count < len(feature_authors) for count in counter_features.values()]


In [17]:
filter_mask = [a*b for a, b in zip(common_mask, notpunct_features)]

In [18]:
filter_mask = [i for i, val in enumerate(filter_mask) if val != 0]

filtered_feature_authors = feature_authors[:, filter_mask]

filtered_feature_names = np.array(feature_names)[filter_mask]

filtered_feature_authors = normalize_array(filtered_feature_authors)

In [19]:
filtered_counter_features = Counter()

for author in filtered_feature_authors:
    for i, feature in enumerate(author):
        if feature != 0:
          filtered_counter_features[i]+=1

In [20]:
print(len(filter_mask))

4089


In [21]:
print(len(feature_authors[0]))

10576


#pure infogain

In [22]:
from sklearn.feature_selection import mutual_info_classif

In [23]:
feature_stats = [filtered_counter_features[i]/len(filtered_counter_features) for i in filtered_counter_features]

In [24]:
classification_feature_info_gain = defaultdict(dict)
parts_vect_np = np.array(writers.parts_vect)
repeat = 1
parts_vect_np = parts_vect_np[:, filter_mask]
parts_x = parts_vect_np.shape[0]
how_separate = 'syntax'
part_len_sep = 'whole'
author_features = dict()

print(f"{'name':18},{'precision':>10},{'recall':>10},{'support':>10}")
for cur_author_name, cur_author_id in tqdm(writers.authors_id.items()):
    print(cur_author_name, end='\r')
    parts_no = parts_author_id_np[parts_author_id_np==cur_author_id].shape[0]

    labels_orig = np.zeros((parts_x))
    labels_orig[parts_author_id_np==cur_author_id] = 1

    labels = np.zeros((parts_x + repeat * parts_no))
    labels[:labels_orig.shape[0]][parts_author_id_np==cur_author_id] = 1
    labels[parts_x:] = 1

    data = np.zeros((parts_x + repeat * parts_no, parts_vect_np.shape[1]))
    data[:parts_x, :] = parts_vect_np

    for rep in range(repeat):
        data[parts_x+parts_no*rep:parts_x+parts_no*(rep+1), :] = parts_vect_np[parts_author_id_np==cur_author_id]

        model = LinearRegression()
        model.fit(data, labels)

        y_hat = model.predict(parts_vect_np)

        y_hat[y_hat>0.5] = 1
        y_hat[y_hat<=0.5] = 0

        res = precision_recall_fscore_support(labels_orig, y_hat, zero_division=0.0)

        info_gain = mutual_info_classif(data, labels)
        # entropy_features = entropy_(labels)

        classification_feature_info_gain[cur_author_id]['info_gain'] = info_gain
        classification_feature_info_gain[cur_author_id]['feature_stats'] = feature_stats
        classification_feature_info_gain[cur_author_id]['features'] = [i for i in np.argsort(np.abs(info_gain))[::-1]]
        # classification_feature_info_gain[cur_author_name]['entropy'] = entropy_features


    print(f"{cur_author_name:18},{res[0][1]:10.3},{res[1][1]:10.3}, {res[3][1]:>4} / {res[3][0]}")

name              , precision,    recall,   support


  0%|          | 0/124 [00:00<?, ?it/s]

  0%|          | 0/124 [00:23<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
_rows = []
for cluster, info in classification_feature_info_gain.items():
      row = {
          'authors': cluster,
          'information_gain': info['info_gain'],
          'feature_stats': info['feature_stats'],
          'features': info['features']
      }
      _rows.append(row)

classification_feature_info_gain_df = pd.DataFrame(_rows)

In [ ]:
classification_feature_info_gain_df.to_csv('only_infogain_ALLFEATS_1000sents.csv')

In [ ]:
classification_feature_info_gain_df.head()

,authors,information_gain,feature_stats,features
0,0,"[0.0, 0.0, 0.0, 0.0, 0.004287192293372266, 0.0...","[0.01189001733960862, 0.007678969531830567, 0....","[389, 673, 3695, 1206, 3579, 2310, 3956, 3946,..."
1,1,"[0.0016377449188518245, 0.011048310308769738, ...","[0.01189001733960862, 0.007678969531830567, 0....","[3672, 3915, 3956, 3742, 319, 2652, 88, 1069, ..."
2,2,"[0.0014461200100530647, 0.0032255852992817413,...","[0.01189001733960862, 0.007678969531830567, 0....","[1602, 3824, 263, 1942, 1618, 1731, 145, 2457,..."
3,3,"[0.0028666349419861348, 0.0040315829396504466,...","[0.01189001733960862, 0.007678969531830567, 0....","[979, 4026, 19, 3956, 1598, 1602, 929, 1266, 3..."
4,4,"[0.0042163859242274615, 2.9430043455347388e-05...","[0.01189001733960862, 0.007678969531830567, 0....","[2791, 3195, 2723, 329, 94, 923, 2118, 3276, 3..."


In [ ]:
feature_names_np = np.array(feature_names)
for idx, row in classification_feature_info_gain_df.iterrows():
    features = row['features']
    info_gain = row['information_gain']
    authors = row['authors']
    stats = row['feature_stats']

    indexed_info_gain = []
    for i, value in enumerate(info_gain):
        indexed_info_gain.append((value, i))

    sorted_indexed_info_gain = sorted(indexed_info_gain, key=lambda item: item[0], reverse=True)

    new_info_gain = [item[0] for item in sorted_indexed_info_gain]
    sorted_features = [item[1] for item in sorted_indexed_info_gain]
    sorted_stats = [stats[i] for i in features]

    classification_feature_info_gain_df.at[idx, 'information_gain'] = new_info_gain[:51]
    classification_feature_info_gain_df.at[idx, 'feature_stats'] = sorted_stats[:51]
    classification_feature_info_gain_df.at[idx, 'features'] = sorted_features[:51]
    classification_feature_info_gain_df.at[idx, 'authors'] = author_id_to_name[authors]
    classification_feature_info_gain_df.at[idx, 'feature_names'] = feature_names_np[np.array(sorted_features)[:51]]


/tmp/ipython-input-1736606071.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Voinovich' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  classification_feature_info_gain_df.at[idx, 'authors'] = author_id_to_name[authors]


In [ ]:
features_to_classify_on = set()

for i, row in classification_feature_info_gain_df.iterrows():
  for feature in row['features']:
        features_to_classify_on.add(feature)

features_to_classify_on = list(features_to_classify_on)

In [ ]:
len(features_to_classify_on)

1076

In [ ]:
classification_feature_info_gain = defaultdict(dict)
parts_vect_np = np.array(writers.parts_vect)
repeat = 1
parts_vect_np = parts_vect_np[:, features_to_classify_on]
parts_x = parts_vect_np.shape[0]
how_separate = 'syntax'
part_len_sep = 'whole'
author_features = dict()

print(f"{'name':18},{'precision':>10},{'recall':>10},{'support':>10}")
with open(f"writers_{how_separate}_{part_len_sep}.csv", "at") as out_file:
    out_file.write(f"{'name':18};{'precision':>10};{'recall':>10};{'f-score':>10};{'support':>10}\n")

for cur_author_name, cur_author_id in writers.authors_id.items():
    print(cur_author_name, end='\r')
    parts_no = parts_author_id_np[parts_author_id_np==cur_author_id].shape[0]

    labels_orig = np.zeros((parts_x))
    labels_orig[parts_author_id_np==cur_author_id] = 1

    labels = np.zeros((parts_x + repeat * parts_no))
    labels[:labels_orig.shape[0]][parts_author_id_np==cur_author_id] = 1
    labels[parts_x:] = 1

    data = np.zeros((parts_x + repeat * parts_no, parts_vect_np.shape[1]))
    data[:parts_x, :] = parts_vect_np

    for rep in range(repeat):
        data[parts_x+parts_no*rep:parts_x+parts_no*(rep+1), :] = parts_vect_np[parts_author_id_np==cur_author_id]

        model = LinearRegression()
        model.fit(data, labels)

        y_hat = model.predict(parts_vect_np)

        y_hat[y_hat>0.5] = 1
        y_hat[y_hat<=0.5] = 0

        res = precision_recall_fscore_support(labels_orig, y_hat, zero_division=0.0)

        # classification_feature_info_gain[cur_author_name]['entropy'] = entropy_features


    print(f"{cur_author_name:18},{res[0][1]:10.3},{res[1][1]:10.3}, {res[3][1]:>4} / {res[3][0]}")

name              , precision,    recall,   support
Voinovich         ,     0.929,     0.333,   39 / 4064
Serafimovich      ,       1.0,     0.586,   70 / 4033
SaltykovSchedrin  ,     0.935,     0.597,   72 / 4031
Vainery           ,     0.967,     0.408,   71 / 4032
Bushkov           ,     0.852,     0.333,   69 / 4034
Kozlov            ,       1.0,     0.103,   39 / 4064
Brustein          ,       1.0,       0.3,   30 / 4073
Fadeev            ,      0.75,    0.0769,   39 / 4064
Bykov             ,       1.0,     0.212,   33 / 4070
Preobrazhensky    ,       1.0,     0.538,   13 / 4090
Koni              ,       0.0,       0.0,    2 / 4101
Zemlyanoj         ,       1.0,     0.458,   72 / 4031
Marinina          ,     0.962,     0.333,   75 / 4028
Furmanov          ,       1.0,       0.6,   35 / 4068
Akunin            ,     0.939,      0.47,   66 / 4037
Belyanin          ,     0.923,       0.6,   20 / 4083
Exler             ,       1.0,     0.294,   17 / 4086
Astafiev          ,       0.9,

#z-score

In [ ]:
authors = []
cluster_feature_info = defaultdict(dict)

def get_info_authors(author_data, author_ids, feature_names, n_clusters = 2):
    clusterisation = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')

    if len(author_ids) > 1:

        cluster_labels = clusterisation.fit_predict(author_data)
        global_feature_means = np.mean(author_data, axis=0)

        for cluster_id in range(n_clusters):

            cluster_mask = (cluster_labels == cluster_id)
            cluster_data = author_data[cluster_mask]

            feature_mask = (cluster_data.sum(axis = 0)!=0)
            author_data2 = author_data[:, feature_mask]
            local_feature_means = global_feature_means[feature_mask]
            feature_names2 = np.array(feature_names)[feature_mask]

            cluster_author_ids = author_ids[cluster_mask]
            cluster_authors = [author_id_to_name[author_id] for author_id in cluster_author_ids]

            cluster_data = cluster_data[:, cluster_data.sum(axis = 0)!=0]

            cluster_means = np.mean(cluster_data, axis=0)
            z_scores = (cluster_means - local_feature_means) / np.std(author_data2, axis=0)

            cluster_feature_info[str(cluster_authors)]['z_score'] = z_scores
            cluster_feature_info[str(cluster_authors)]['features'] = [i for i in np.argsort(np.abs(z_scores))[::-1][:51]]
            cluster_feature_info[str(cluster_authors)]['feature_stats'] = feature_stats

            if len(authors) == len(author_data):
                return cluster_feature_info
            else:
                cluster_mask = (cluster_labels == cluster_id)
                subcluster_data = author_data2[cluster_mask]
                sub_author_ids = author_ids[cluster_mask]
                get_info_authors(subcluster_data, sub_author_ids, feature_names2)

    elif len(author_ids) == 1:
        authors.append(author_ids)


In [ ]:
# from sklearn.cluster import KMeans

# def get_info_authors(author_data, author_ids, feature_names, n_clusters=2):
#     if len(author_ids) <= 1:
#         authors.append(author_ids)
#         return

#     # Top-down split: one cluster → n_clusters subgroups
#     kmeans = KMeans(n_clusters=min(n_clusters, len(author_ids)), random_state=42)
#     cluster_labels = kmeans.fit_predict(author_data)

#     # Process each subcluster (your existing cluster analysis code)
#     for cluster_id in range(max(cluster_labels) + 1):
#         cluster_mask = (cluster_labels == cluster_id)
#         subcluster_data = author_data[cluster_mask]
#         sub_author_ids = np.array(author_ids)[cluster_mask]

#         # Your existing feature analysis...
#         cluster_feature_info[str([author_id_to_name[a] for a in sub_author_ids])]['z_score'] = z_scores

#         # Recurse
#         get_info_authors(subcluster_data, sub_author_ids, feature_names)


In [ ]:
final_info = get_info_authors(filtered_feature_authors, author_ids, filtered_feature_names)

In [ ]:
_rows = []
for cluster, info in final_info.items():
      row = {
          'authors': cluster,
          'z_score': info['z_score'],
          'features': info['features'],
          'feature_stats': info['feature_stats']
      }
      _rows.append(row)

data = pd.DataFrame(_rows)
data.head()

AttributeError: 'NoneType' object has no attribute 'items'

In [ ]:
import ast
data['authors'] = data['authors'].apply(lambda x: ast.literal_eval(x))

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
new_features_after_clustering_df = data[data['authors'].map(len) == 1]

new_features_after_clustering_df = new_features_after_clustering_df.map(lambda x: x[0] if isinstance(x, tuple) and len(x) > 0 else x)
new_features_after_clustering_df['features'] = new_features_after_clustering_df['features'].apply(lambda x: x[:51])


In [ ]:
new_features_after_clustering_df.head()

,authors,z_score,features,feature_stats
4,[AkuninBrusnikin],"[1.0, 1.0, 1.0, 1.0, -1.0000000000000002, -1.0...","[422, 957, 226, 81, 1127, 1223, 523, 125, 106,...","[0.0009101941747572816, 0.00030339805825242716..."
5,[Akunin],"[-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0...","[642, 1440, 1687, 804, 160, 258, 1385, 906, 22...","[0.0009101941747572816, 0.00030339805825242716..."
6,[Abgaryan],"[1.3843318402998395, 1.334524481160197, 1.4142...","[91, 87, 462, 67, 604, 267, 245, 39, 587, 600,...","[0.0009101941747572816, 0.00030339805825242716..."
7,[Adamov],"[0.8422824921884743, 0.9116393479196916, 1.627...","[426, 72, 952, 103, 173, 149, 380, 689, 318, 8...","[0.0009101941747572816, 0.00030339805825242716..."
8,[AkuninBorisova],"[1.9528194605200553, 1.9943135332809017, 1.846...","[49, 44, 29, 99, 53, 77, 62, 83, 98, 79, 10, 2...","[0.0009101941747572816, 0.00030339805825242716..."


In [ ]:
for idx, row in new_features_after_clustering_df.iterrows():
    features = row['features']
    info_gain = row['z_score']
    authors = row['authors']
    stats = row['feature_stats']

    indexed_info_gain = []
    for i, value in enumerate(info_gain):
        indexed_info_gain.append((value, i))

    sorted_indexed_info_gain = sorted(indexed_info_gain, key=lambda item: item[0], reverse=True)

    new_info_gain = [item[0] for item in sorted_indexed_info_gain]
    sorted_features = [item[1] for item in sorted_indexed_info_gain]
    sorted_stats = [stats[i] for i in features]

    new_features_after_clustering_df.at[idx, 'z_score'] = new_info_gain[:51]
    new_features_after_clustering_df.at[idx, 'features'] = sorted_features[:51]
    new_features_after_clustering_df.at[idx, 'feature_stats'] = sorted_stats[:51]
    new_features_after_clustering_df.at[idx, 'feature_names'] = np.array(feature_names)[sorted_features[:51]]


In [ ]:
new_features_after_clustering_df.head()

,authors,z_score,features,feature_stats,feature_names
4,[AkuninBrusnikin],"[1.0000000000000189, 1.0000000000000053, 1.000...","[81, 523, 125, 1169, 1324, 1360, 91, 1064, 754...","[0.0006067961165048543, 0.0006067961165048543,...","[(INTJ, <-, NOUN, nsubj), (NOUN, <-, PUNCT, PA..."
5,[Akunin],"[1.0000000000000366, 1.0000000000000042, 1.000...","[1687, 1157, 464, 931, 1873, 796, 1601, 315, 1...","[0.001516990291262136, 0.001516990291262136, 0...","[(PROPN, <-, PROPN, nummod:gov), (ADV, ->, PRO..."
6,[Abgaryan],"[1.4142135623730954, 1.4142135623730954, 1.414...","[39, 67, 87, 91, 179, 245, 267, 462, 587, 600,...","[0.0006067961165048543, 0.0009101941747572816,...","[(ADV, <-, ADV, case), (VERB, ->, ADV, advmod)..."
7,[Adamov],"[1.7320508075688774, 1.7320508075688774, 1.732...","[28, 41, 48, 67, 72, 85, 103, 149, 170, 173, 1...","[0.0012135922330097086, 0.00030339805825242716...","[(X, ->, NOUN, nsubj), (NOUN, <-, PART, orphan..."
8,[AkuninBorisova],"[2.0000000000000004, 2.0, 2.0, 2.0, 2.0, 2.0, ...","[49, 10, 13, 18, 29, 44, 53, 62, 77, 79, 83, 9...","[0.00030339805825242716, 0.0006067961165048543...","[(VERB, <-, NOUN, fixed), (VERB, <-, PART, dis..."


In [ ]:
new_features_after_clustering_df.to_csv('z_score_6authors.csv')

In [ ]:
features_to_classify_on = set()

for i, row in new_features_after_clustering_df.iterrows():
  for feature in row['features']:
        features_to_classify_on.add(feature)

features_to_classify_on = list(features_to_classify_on)
print(features_to_classify_on)

[1025, 3, 1028, 515, 6, 517, 7, 1545, 522, 523, 524, 10, 13, 11, 528, 529, 18, 20, 22, 538, 26, 28, 29, 543, 31, 1058, 35, 1063, 1064, 39, 554, 43, 44, 41, 558, 40, 560, 48, 49, 45, 53, 1079, 56, 1081, 58, 571, 59, 57, 62, 61, 1601, 578, 67, 65, 69, 1096, 72, 586, 587, 73, 77, 1102, 79, 80, 81, 593, 83, 84, 85, 1110, 87, 600, 599, 88, 91, 604, 605, 93, 1119, 610, 99, 98, 614, 103, 108, 114, 116, 117, 2166, 118, 120, 122, 125, 128, 641, 129, 1155, 1157, 139, 140, 141, 1169, 146, 147, 1684, 149, 662, 1687, 150, 665, 153, 152, 669, 159, 160, 675, 1188, 164, 168, 1193, 170, 172, 685, 173, 686, 176, 689, 178, 179, 177, 182, 696, 185, 186, 1724, 189, 190, 703, 196, 1220, 198, 199, 201, 202, 203, 205, 1233, 211, 212, 217, 218, 221, 1249, 228, 229, 741, 1255, 742, 746, 234, 236, 1261, 750, 239, 235, 241, 754, 755, 244, 245, 247, 759, 761, 250, 763, 1276, 764, 766, 767, 252, 257, 260, 773, 778, 267, 269, 782, 272, 277, 278, 789, 280, 283, 796, 284, 286, 798, 288, 801, 292, 295, 807, 1319, 1323,

In [ ]:
len(features_to_classify_on)

274

In [ ]:
classification_feature_info_gain = defaultdict(dict)
parts_vect_np = np.array(writers.parts_vect)
repeat = 1
parts_vect_np = parts_vect_np[:, features_to_classify_on]
parts_x = parts_vect_np.shape[0]
how_separate = 'syntax'
part_len_sep = 'whole'
author_features = dict()

print(f"{'name':18},{'precision':>10},{'recall':>10},{'support':>10}")
with open(f"writers_{how_separate}_{part_len_sep}.csv", "at") as out_file:
    out_file.write(f"{'name':18};{'precision':>10};{'recall':>10};{'f-score':>10};{'support':>10}\n")

for cur_author_name, cur_author_id in writers.authors_id.items():
    print(cur_author_name, end='\r')
    parts_no = parts_author_id_np[parts_author_id_np==cur_author_id].shape[0]

    labels_orig = np.zeros((parts_x))
    labels_orig[parts_author_id_np==cur_author_id] = 1

    labels = np.zeros((parts_x + repeat * parts_no))
    labels[:labels_orig.shape[0]][parts_author_id_np==cur_author_id] = 1
    labels[parts_x:] = 1

    data = np.zeros((parts_x + repeat * parts_no, parts_vect_np.shape[1]))
    data[:parts_x, :] = parts_vect_np

    for rep in range(repeat):
        data[parts_x+parts_no*rep:parts_x+parts_no*(rep+1), :] = parts_vect_np[parts_author_id_np==cur_author_id]

        model = LinearRegression()
        model.fit(data, labels)

        y_hat = model.predict(parts_vect_np)

        y_hat[y_hat>0.5] = 1
        y_hat[y_hat<=0.5] = 0

        res = precision_recall_fscore_support(labels_orig, y_hat, zero_division=0.0)

        # classification_feature_info_gain[cur_author_name]['entropy'] = entropy_features


    print(f"{cur_author_name:18},{res[0][1]:10.3},{res[1][1]:10.3}, {res[3][1]:>4} / {res[3][0]}")

name              , precision,    recall,   support
Akunin            ,     0.697,     0.846,  599 / 994
Aksakov           ,     0.755,     0.757,  325 / 1268
Abgaryan          ,     0.733,     0.198,  111 / 1482
AkuninBrusnikin   ,      0.61,     0.479,  219 / 1374
Adamov            ,     0.686,     0.807,  290 / 1303
AkuninBorisova    ,       1.0,     0.265,   49 / 1544


# Information gain after z-score

In [ ]:
authors = []
cluster_feature_info = defaultdict(dict)

def get_info_features(author_data, author_ids, feature_names, n_clusters = 2):
    clusterisation = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')

    if len(author_ids) > 1:

        cluster_labels = clusterisation.fit_predict(author_data)
        global_feature_means = np.mean(author_data, axis=0)

        for cluster_id in range(n_clusters):

            cluster_mask = (cluster_labels == cluster_id)
            cluster_data = author_data[cluster_mask]

            feature_mask = (cluster_data.sum(axis = 0)!=0)
            author_data2 = author_data[:, feature_mask]
            local_feature_means = global_feature_means[feature_mask]
            feature_names2 = np.array(feature_names)[feature_mask]

            cluster_author_ids = tuple(author_ids[cluster_mask])
            # cluster_authors = [author_id_to_name[author_id] for author_id in cluster_author_ids]

            cluster_data = cluster_data[:, cluster_data.sum(axis = 0)!=0]

            cluster_means = np.mean(cluster_data, axis=0)
            z_scores = (cluster_means - local_feature_means) / np.std(author_data2, axis=0)

            cluster_feature_info[cluster_author_ids] = [i for i in np.argsort(np.abs(z_scores))[::-1][:101]]

            if len(authors) == 109:
                return cluster_feature_info
            else:
                cluster_mask = (cluster_labels == cluster_id)
                subcluster_data = author_data2[cluster_mask]
                sub_author_ids = author_ids[cluster_mask]
                get_info_features(subcluster_data, sub_author_ids, feature_names2)

    elif len(author_ids) == 1:
        authors.append(author_ids)


In [ ]:
features_after_clustering =  (get_info_features(filtered_feature_authors, author_ids, filtered_feature_names))

In [ ]:
features_after_clustering_df = pd.DataFrame({'authors': features_after_clustering.keys(),
                     'Features': features_after_clustering.values()})
features_after_clustering_df.head()

,authors,Features
0,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[348, 183, 338, 133, 383, 189, 122, 159, 262, ..."
1,"(16, 25, 27, 29, 30, 34, 35, 37, 42, 50, 59, 7...","[310, 66, 180, 136, 46, 229, 347, 10, 388, 415..."
2,"(16, 25, 29, 30, 34, 35, 37, 42, 59, 77, 81, 8...","[165, 211, 357, 84, 3, 9, 224, 186, 297, 87, 1..."
3,"(16, 25, 29, 30, 34, 37, 42, 59, 77, 83, 95, 1...","[315, 8, 348, 110, 262, 188, 183, 95, 122, 196..."
4,"(30, 37, 42, 77, 101)","[179, 92, 4, 289, 113, 131, 6, 379, 175, 312, ..."


In [ ]:
features_to_classify_on = set()

for values in features_after_clustering.values():
  for feature in values:
        features_to_classify_on.add(feature)

features_to_classify_on = list(features_to_classify_on)

In [ ]:
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif

# def entropy_(labels, base=None):
#   value, counts = np.unique(labels, return_counts=True)
#   return entropy(counts, base=base)

In [ ]:
classification_feature_info_gain = defaultdict(dict)
parts_vect_np = np.array(writers.parts_vect)
parts_author_id_np = np.array(writers.parts_author_id)

repeat = 1
parts_vect_np = parts_vect_np[:, features_to_classify_on]
parts_x = parts_vect_np.shape[0]

In [ ]:
for cur_author_name, cur_author_id in writers.authors_id.items():
    print(cur_author_name, end='\r')
    parts_no = parts_author_id_np[parts_author_id_np==cur_author_id].shape[0]

    labels_orig = np.zeros((parts_x))
    labels_orig[parts_author_id_np==cur_author_id] = 1

    labels = np.zeros((parts_x + repeat * parts_no))
    labels[:labels_orig.shape[0]][parts_author_id_np==cur_author_id] = 1
    labels[parts_x:] = 1

    data = np.zeros((parts_x + repeat * parts_no, parts_vect_np.shape[1]))
    data[:parts_x, :] = parts_vect_np

    for rep in range(repeat):
        data[parts_x+parts_no*rep:parts_x+parts_no*(rep+1), :] = parts_vect_np[parts_author_id_np==cur_author_id]

        model = LinearRegression()
        model.fit(data, labels)

        y_hat = model.predict(parts_vect_np)

        y_hat[y_hat>0.5] = 1
        y_hat[y_hat<=0.5] = 0

        res = precision_recall_fscore_support(labels_orig, y_hat, zero_division=0.0)

        info_gain = mutual_info_classif(data, labels)
        # entropy_features = entropy_(labels)

        classification_feature_info_gain[cur_author_id]['info_gain'] = info_gain
        # classification_feature_info_gain[cur_author_name]['entropy'] = entropy_features


    print(f"{cur_author_name:18},{res[0][1]:10.3},{res[1][1]:10.3}, {res[3][1]:>4} / {res[3][0]}")

Cherkasova        ,     0.778,     0.184,  190 / 6727
Dragunski         ,     0.714,    0.0746,   67 / 6850
Grekova           ,     0.923,     0.167,   72 / 6845
Shukshin          ,      0.85,     0.337,  101 / 6816
Zemlyanoj         ,     0.792,     0.104,  182 / 6735
Kozlov            ,      0.75,    0.0319,   94 / 6823
Prokofeva         ,      0.75,    0.0857,   35 / 6882
Abgaryan          ,      0.75,     0.107,   28 / 6889
Strugatskie       ,     0.577,     0.115,  131 / 6786
Derevyanko        ,     0.769,     0.172,   58 / 6859
Belyanin          ,       0.8,     0.114,   35 / 6882
AkuninBrusnikin   ,     0.846,     0.164,   67 / 6850
Aleshkovsky       ,     0.714,    0.0877,   57 / 6860
Akunin            ,     0.771,     0.191,  141 / 6776
Dostoevsky        ,     0.924,     0.477,  281 / 6636
Makushinskiyi     ,       1.0,     0.231,   26 / 6891
Gusev             ,       1.0,    0.0625,   16 / 6901
Tolstoy           ,     0.899,     0.652,  313 / 6604
Volkov            ,       0.

In [ ]:
_rows = []
for cluster, info in classification_feature_info_gain.items():
      row = {
          'authors': cluster,
          'information_gain': info['info_gain']
      }
      _rows.append(row)

classification_feature_info_gain_df = pd.DataFrame(_rows)
classification_feature_info_gain_df.head()

,authors,information_gain
0,0,"[0.0, 0.0, 0.0, 0.0, 0.01234532357157625, 0.00..."
1,1,"[0.007480784112675054, 0.0019611640094101546, ..."
2,2,"[0.0, 0.0, 0.0211237684775889, 0.0, 0.00256667..."
3,3,"[0.0, 0.009076410586321693, 0.0041072778427626..."
4,4,"[0.002918312590842964, 0.005383427912197369, 0..."


In [ ]:
new_features_after_clustering_df = features_after_clustering_df[features_after_clustering_df['authors'].map(len) == 1]

new_features_after_clustering_df = new_features_after_clustering_df.map(lambda x: x[0] if isinstance(x, tuple) and len(x) > 0 else x)
new_features_after_clustering_df['Features'] = new_features_after_clustering_df['Features'].apply(lambda x: x[:5])

In [ ]:
merged_df = pd.merge(new_features_after_clustering_df, classification_feature_info_gain_df, on='authors')

merged_df.head()

,authors,Features,information_gain
0,45,"[42, 210, 65, 196, 194]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.001052707671447938..."
1,25,"[266, 139, 264, 195, 128]","[0.0, 0.011711539615835265, 0.0058058298478501..."
2,73,"[4, 5, 66, 53, 52]","[0.0, 0.0003145778765141216, 0.002006395286274..."
3,95,"[123, 193, 105, 227, 261]","[0.002402297551933552, 0.0023440709633575363, ..."
4,90,"[161, 93, 249, 264, 334]","[0.0, 0.0, 0.0, 0.023460025048530286, 0.0, 0.0..."


#на этих фичах построить классификацию еще раз и посмотреть результат лучше хуже

In [ ]:
# for idx, row in merged_df.iterrows():
#     features = row['Features']
#     info_gain = row['information_gain']
#     authors = row['authors']

#     new_info_gain = [info_gain[i] for i in features]

#     merged_df.at[idx, 'information_gain'] = new_info_gain
#     merged_df.at[idx, 'authors'] = author_id_to_name[authors]
#     merged_df.at[idx, 'feature_names'] = feature_names[features]

for idx, row in merged_df.iterrows():
    features = row['Features']
    info_gain = row['information_gain']
    authors = row['authors']

    indexed_info_gain = []
    for i, value in enumerate(info_gain):
        indexed_info_gain.append((value, i))

    sorted_indexed_info_gain = sorted(indexed_info_gain, key=lambda item: item[0], reverse=True)

    new_info_gain = [item[0] for item in sorted_indexed_info_gain]
    sorted_features = [item[1] for item in sorted_indexed_info_gain]

    merged_df.at[idx, 'information_gain'] = new_info_gain[:5]
    merged_df.at[idx, 'Features'] = sorted_features[:5]
    merged_df.at[idx, 'authors'] = author_id_to_name[authors]
    merged_df.at[idx, 'feature_names'] = feature_names[sorted_features[:5]]



/tmp/ipython-input-921234068.py:28: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Chechin' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



In [ ]:
merged_df.head()

,authors,Features,information_gain,feature_names
0,Chechin,"[247, 261, 249, 285, 62]","[0.028372305750285598, 0.028372305750285598, 0...","[(NUM, <-, NUM, mark), (NUM, ->, ADP, obl), (V..."
1,Kirsanova Anodina Aksenov,"[75, 153, 159, 53, 127]","[0.028372305750285598, 0.028372305750285598, 0...","[(CCONJ, ->, ADV, advmod), (CCONJ, <-, DET, ns..."
2,nlobooks[ru],"[95, 156, 170, 249, 321]","[0.028372305750285598, 0.028372305750285598, 0...","[(VERB, <-, PROPN, iobj), (NOUN, ->, X, conj),..."
3,Gusev,"[52, 249, 239, 303, 369]","[0.043530143335480886, 0.043530143335480775, 0...","[(ADV, <-, VERB, acl), (VERB, ->, SCONJ, fixed..."
4,Semihatov,"[13, 45, 62, 37, 357]","[0.02837230575028471, 0.027144235574846354, 0....","[(NUM, ->, ADV, acl), (ADV, <-, PART, det), (A..."


In [ ]:
merged_df.to_csv('z_score_then_infogain_results.csv')

z-score добавить в таблицу выше

In [ ]:
def visualize_authors_radar(df, feature_counter):

    fig = go.Figure()

    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']

    for i, row in df.iterrows():
        author = row['authors']
        features = row['Features']
        info_gain = row['information_gain']
        feature_names = row['feature_names']

        min_length = min(len(info_gain), len(feature_names), len(features))
        features = features[:min_length]
        info_gain = info_gain[:min_length]
        feature_names = feature_names[:min_length]

        values = np.abs(info_gain)

        custom_hover_data = []
        for feature_idx, feature_name, ig_value in zip(features, feature_names, info_gain):
            frequency = feature_counter[feature_idx]
            custom_hover_data.append(
                f"Freq: {frequency} | ID: {feature_idx} | IG: {ig_value:.6f}"
            )

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=feature_names,
            fill='toself',
            name=author,
            hovertemplate='<b>Author:</b> ' + author + '<br>' +
                         '<b>Feature:</b> %{theta}<br>' +
                         '<b>Info Gain:</b> %{r:.6f}<br>' +
                         '<b>Details:</b> %{customdata}<br>' +
                         '<extra></extra>',
            customdata=custom_hover_data,
            line=dict(color=colors[i % len(colors)]),
            opacity=0.7
        ))

    max_val = 0
    for _, row in df.iterrows():
        if len(row['information_gain']) > 0:
            max_val = max(max_val, max(np.abs(row['information_gain'])))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, max_val * 1.1] if max_val > 0 else [0, 1]
            )
        ),
        title=dict(
            text=str(list(df['authors'])),
            x=0.5,
            xanchor='center'
        ),
        showlegend=True,
        width=800,
        height=600
    )

    fig.show()

In [ ]:
filtered_counter_features = Counter()
for author in filtered_feature_authors:
    for i, feature in enumerate(author):
        if feature > 0.006:
          filtered_counter_features[i]+=1
        else:
          filtered_counter_features[i] += 0

In [ ]:
author_lists_clustered = features_after_clustering_df[features_after_clustering_df['authors'].apply(len).between(4, 7)]['authors'].tolist()

In [ ]:
selected_authors = []
for cluster in author_lists_clustered:
    author_names_of_cluster = []
    for id in cluster:
        author_names_of_cluster.append(author_id_to_name[id])
    selected_authors.append(tuple(author_names_of_cluster))

#info gain visualisation


In [ ]:
for i, cluster in enumerate(selected_authors):
    visualize_authors_radar(merged_df[merged_df['authors'].isin(selected_authors[i])], filtered_counter_features)